# 05 — ML Model: Predictive Hiring with MLflow

**Jackson and Jackson HR Digital**

This notebook trains three candidate-hiring classifiers against Jackson and Jackson's historical scoring data,
compares results in MLflow, registers the best model to Unity Catalog, and deploys a
real-time Model Serving endpoint.

### What this notebook does
1. Loads training-eligible candidates from the `candidates` Silver table (C001–C010, all 8 features complete)
2. Engineers features using 8 scoring dimensions and performs a 70/30 train/test split
3. Trains **LogisticRegression**, **RandomForestClassifier**, and **GradientBoostingClassifier** — each tracked in a dedicated MLflow run
4. Compares AUC-ROC across runs and identifies the best model
5. Registers the winning model to Unity Catalog under the `@champion` alias
6. Deploys a Model Serving endpoint (scale-to-zero, `Small` workload size)
7. Tests inference with a sample candidate (Sarah Chen)

### Data Model
- **Training set**: C001–C010 (10 candidates with complete feature scores + `hired` label)
- **Inference candidates**: C011–C020 (10 new candidates — `skills_match_score`, `interview_score`, `culture_fit`, and `hired` are NULL until after their interviews)
- **8 features**: `education_score`, `experience_score`, `leadership_score`, `certification_score`, `skills_match_score`, `industry_relevance_score`, `interview_score`, `culture_fit`

### Prerequisites
- Notebooks `00` – `03` have been run (Silver tables exist with 20 candidates)
- MLflow is available on the cluster (bundled with Databricks Runtime 13.x+)
- Unity Catalog enabled workspace with `CREATE MODEL` privilege on the target schema

**Next notebook:** `06_evaluate_register_agent.ipynb`

## Setup

Install dependencies and configure catalog, schema, MLflow experiment, and serving endpoint names.

In [ ]:
%pip install scikit-learn mlflow databricks-sdk -q

In [ ]:
dbutils.library.restartPython()

In [ ]:
# ---------------------------------------------------------------------------
# Widget definitions — edit defaults here or override at runtime via the UI
# ---------------------------------------------------------------------------
dbutils.widgets.text("catalog",           "bx4",                          "UC Catalog")
dbutils.widgets.text("schema",            "hrd_2030",                     "UC Schema")
dbutils.widgets.text("mlflow_experiment", "hr_predictive_hiring",         "MLflow Experiment Name")
dbutils.widgets.text("model_name",        "hr_predictive_hiring",         "UC Model Name")
dbutils.widgets.text("endpoint_name",     "hr-predictive-hiring-endpoint","Serving Endpoint Name")

In [ ]:
import os
import mlflow
import pandas as pd
import numpy as np
from pprint import pprint
from databricks.sdk import WorkspaceClient

# Load .env for interactive runs — no-op if file doesn't exist
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
project_root  = "/Workspace" + notebook_path.rsplit("/notebooks", 1)[0]
try:
    from dotenv import load_dotenv
    load_dotenv(f"{project_root}/.env")
except ImportError:
    pass

# Widget takes priority (job passes it); fall back to .env for interactive runs
catalog           = dbutils.widgets.get("catalog")           or os.getenv("TARGET_CATALOG", "bx4")
schema            = dbutils.widgets.get("schema")            or os.getenv("TARGET_SCHEMA", "hrd_2030")
mlflow_experiment = dbutils.widgets.get("mlflow_experiment") or os.getenv("MLFLOW_EXPERIMENT_ML", "ml-hr-predictive-hiring")
model_name        = dbutils.widgets.get("model_name")
endpoint_name     = dbutils.widgets.get("endpoint_name")     or os.getenv("MODEL_ENDPOINT_NAME", "hr-predictive-hiring-endpoint")

# Resolve current user for experiment path
current_user = spark.sql("SELECT current_user()").collect()[0][0]
experiment_path = f"/Users/{current_user}/{mlflow_experiment}"

print(f"Catalog           : {catalog}")
print(f"Schema            : {schema}")
print(f"MLflow experiment : {experiment_path}")
print(f"Model name        : {catalog}.{schema}.{model_name}")
print(f"Endpoint name     : {endpoint_name}")

## Load Training Data

Read the `candidates` Silver table and filter to the 10 training-eligible candidates (C001–C010) who have complete feature scores and a hire label.

In [ ]:
# ---------------------------------------------------------------------------
# Load training-eligible candidates from the Silver Delta table.
# New candidates (C011–C020) have NULL skills_match_score, interview_score,
# culture_fit, and hired — they are excluded here and used for inference only.
# ---------------------------------------------------------------------------
from pyspark.sql import functions as F

all_candidates = spark.table(f"{catalog}.{schema}.candidates")

training_df = all_candidates.filter(
    F.col("interview_score").isNotNull() &
    F.col("skills_match_score").isNotNull() &
    F.col("culture_fit").isNotNull() &
    F.col("hired").isNotNull()
)

total     = all_candidates.count()
train_cnt = training_df.count()
new_cnt   = total - train_cnt
print(f"Total candidates  : {total}")
print(f"Training-eligible : {train_cnt} (complete features)")
print(f"New / unscored    : {new_cnt} (will be predicted at inference time)")
print("\nTraining set sample:")
display(training_df.select(
    "candidate_id", "first_name", "last_name",
    "education_score", "experience_score", "leadership_score",
    "certification_score", "skills_match_score", "industry_relevance_score",
    "interview_score", "culture_fit", "hired"
).limit(5))

## Feature Engineering & Train/Test Split

Convert to pandas, select the 8 scoring dimensions as features, and split 70/30 with stratification to preserve the hire/no-hire class ratio.

In [ ]:
# ---------------------------------------------------------------------------
# Feature engineering and train/test split
# ---------------------------------------------------------------------------
from sklearn.model_selection import train_test_split

# Convert to pandas for sklearn
df = training_df.toPandas()

# All eight scoring dimensions used as model features
feature_cols = [
    "education_score",
    "experience_score",
    "leadership_score",
    "certification_score",
    "skills_match_score",
    "industry_relevance_score",
    "interview_score",
    "culture_fit",
]
target_col = "hired"

# Ensure numeric types
df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors="coerce")
df[target_col]   = df[target_col].astype(int)

X = df[feature_cols]
y = df[target_col]

# 70/30 split, stratified to preserve class ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Training samples : {len(X_train)}")
print(f"Test samples     : {len(X_test)}")
print(f"Class distribution (train): {y_train.value_counts().to_dict()}")
print(f"Class distribution (test) : {y_test.value_counts().to_dict()}")
print(f"\nFeatures ({len(feature_cols)}): {feature_cols}")

In [ ]:
# ---------------------------------------------------------------------------
# Configure MLflow experiment
# ---------------------------------------------------------------------------
import mlflow

mlflow.set_experiment(experiment_path)

# Enable autolog: captures params, metrics, and the model artifact automatically
mlflow.sklearn.autolog(
    log_input_examples=True,
    log_model_signatures=True,
    silent=True,
)

print(f"\u2705 MLflow experiment set to: {experiment_path}")

## Train Three Models with MLflow Tracking

Train LogisticRegression, RandomForest, and GradientBoosting classifiers. Each run is tracked in MLflow with accuracy, AUC-ROC, and F1 metrics.

In [ ]:
# ---------------------------------------------------------------------------
# Model 1: Logistic Regression
# ---------------------------------------------------------------------------
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

with mlflow.start_run(run_name="LogisticRegression") as run:
    pipeline_lr = Pipeline([
        ("scaler", StandardScaler()),
        ("model",  LogisticRegression(random_state=42, max_iter=200, C=1.0)),
    ])
    pipeline_lr.fit(X_train, y_train)

    y_pred_lr = pipeline_lr.predict(X_test)
    y_prob_lr = pipeline_lr.predict_proba(X_test)[:, 1]

    acc_lr = accuracy_score(y_test, y_pred_lr)
    auc_lr = roc_auc_score(y_test, y_prob_lr)
    f1_lr  = f1_score(y_test, y_pred_lr, zero_division=0)

    mlflow.log_metrics({"accuracy": acc_lr, "auc_roc": auc_lr, "f1": f1_lr})
    lr_run_id = run.info.run_id

print(f"LogisticRegression — Accuracy: {acc_lr:.3f} | AUC-ROC: {auc_lr:.3f} | F1: {f1_lr:.3f}")
print(f"Run ID: {lr_run_id}")

In [ ]:
# ---------------------------------------------------------------------------
# Model 2: Random Forest Classifier
# ---------------------------------------------------------------------------
from sklearn.ensemble import RandomForestClassifier

with mlflow.start_run(run_name="RandomForestClassifier") as run:
    pipeline_rf = Pipeline([
        ("scaler", StandardScaler()),
        ("model",  RandomForestClassifier(
            n_estimators=100,
            max_depth=5,
            random_state=42,
            class_weight="balanced",
        )),
    ])
    pipeline_rf.fit(X_train, y_train)

    y_pred_rf = pipeline_rf.predict(X_test)
    y_prob_rf = pipeline_rf.predict_proba(X_test)[:, 1]

    acc_rf = accuracy_score(y_test, y_pred_rf)
    auc_rf = roc_auc_score(y_test, y_prob_rf)
    f1_rf  = f1_score(y_test, y_pred_rf, zero_division=0)

    mlflow.log_metrics({"accuracy": acc_rf, "auc_roc": auc_rf, "f1": f1_rf})

    # Log feature importances as a custom artifact
    importances = pd.Series(
        pipeline_rf.named_steps["model"].feature_importances_,
        index=feature_cols,
    ).sort_values(ascending=False)
    mlflow.log_dict(importances.to_dict(), "feature_importances.json")

    rf_run_id = run.info.run_id

print(f"RandomForestClassifier — Accuracy: {acc_rf:.3f} | AUC-ROC: {auc_rf:.3f} | F1: {f1_rf:.3f}")
print(f"Run ID: {rf_run_id}")
print(f"\nFeature Importances:")
print(importances.to_string())

In [ ]:
# ---------------------------------------------------------------------------
# Model 3: Gradient Boosting Classifier
# ---------------------------------------------------------------------------
from sklearn.ensemble import GradientBoostingClassifier

with mlflow.start_run(run_name="GradientBoostingClassifier") as run:
    pipeline_gb = Pipeline([
        ("scaler", StandardScaler()),
        ("model",  GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=3,
            random_state=42,
        )),
    ])
    pipeline_gb.fit(X_train, y_train)

    y_pred_gb = pipeline_gb.predict(X_test)
    y_prob_gb = pipeline_gb.predict_proba(X_test)[:, 1]

    acc_gb = accuracy_score(y_test, y_pred_gb)
    auc_gb = roc_auc_score(y_test, y_prob_gb)
    f1_gb  = f1_score(y_test, y_pred_gb, zero_division=0)

    mlflow.log_metrics({"accuracy": acc_gb, "auc_roc": auc_gb, "f1": f1_gb})

    # Log feature importances as a custom artifact
    gb_importances = pd.Series(
        pipeline_gb.named_steps["model"].feature_importances_,
        index=feature_cols,
    ).sort_values(ascending=False)
    mlflow.log_dict(gb_importances.to_dict(), "feature_importances.json")

    gb_run_id = run.info.run_id

print(f"GradientBoostingClassifier — Accuracy: {acc_gb:.3f} | AUC-ROC: {auc_gb:.3f} | F1: {f1_gb:.3f}")
print(f"Run ID: {gb_run_id}")

## Select Best Model by AUC-ROC

Query MLflow to compare all runs and identify the model with the highest AUC-ROC score.

In [ ]:
# ---------------------------------------------------------------------------
# Compare all runs and select the best model by AUC-ROC
# ---------------------------------------------------------------------------
import mlflow

experiment = mlflow.get_experiment_by_name(experiment_path)
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.auc_roc DESC"],
)

display_cols = [
    "tags.mlflow.runName",
    "metrics.accuracy",
    "metrics.auc_roc",
    "metrics.f1",
    "run_id",
]
available_cols = [c for c in display_cols if c in runs.columns]

print("=" * 70)
print("MODEL COMPARISON (ordered by AUC-ROC, descending)")
print("=" * 70)
print(runs[available_cols].rename(columns={
    "tags.mlflow.runName": "Model",
    "metrics.accuracy":    "Accuracy",
    "metrics.auc_roc":     "AUC-ROC",
    "metrics.f1":          "F1",
}).to_string(index=False))
print("=" * 70)

best_run       = runs.iloc[0]
best_run_id    = best_run["run_id"]
best_model_name = best_run.get("tags.mlflow.runName", "UnknownModel")
best_auc       = best_run["metrics.auc_roc"]

print(f"\n\u2705 Best model: {best_model_name}")
print(f"   AUC-ROC  : {best_auc:.3f}")
print(f"   Run ID   : {best_run_id}")

## Register to Unity Catalog

Register the best model to Unity Catalog and assign the `@champion` alias so serving endpoints and agents always reference the current production version.

In [ ]:
# ---------------------------------------------------------------------------
# Register the best model to Unity Catalog and set @champion alias
# ---------------------------------------------------------------------------
import mlflow

# Point MLflow at the Unity Catalog model registry
mlflow.set_registry_uri("databricks-uc")

uc_model_name = f"{catalog}.{schema}.{model_name}"
model_uri     = f"runs:/{best_run_id}/model"

print(f"Registering model from run {best_run_id} to {uc_model_name} ...")
registered_model = mlflow.register_model(
    model_uri=model_uri,
    name=uc_model_name,
)

print(f"  Registered version : {registered_model.version}")

# Set the @champion alias so serving endpoints and agents can always
# reference the current production version by alias rather than version number
client = mlflow.tracking.MlflowClient()
client.set_registered_model_alias(
    name=uc_model_name,
    alias="champion",
    version=registered_model.version,
)

print(f"\n\u2705 Model registered: {uc_model_name}")
print(f"   Version : {registered_model.version}")
print(f"   Alias   : @champion")

# Persist version for the serving cell below
model_version = registered_model.version

## Deploy Model Serving Endpoint

Create or update a scale-to-zero Model Serving endpoint for real-time hire/no-hire predictions.

In [ ]:
# ---------------------------------------------------------------------------
# Deploy a Model Serving endpoint for real-time inference
# Scale-to-zero is enabled to minimise cost when idle.
# ---------------------------------------------------------------------------
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    ServedEntityInput,
    EndpointCoreConfigInput,
)

w = WorkspaceClient()
uc_model_name = f"{catalog}.{schema}.{model_name}"

served_entity = ServedEntityInput(
    entity_name=uc_model_name,
    entity_version=str(model_version),
    scale_to_zero_enabled=True,
    workload_size="Small",
)

endpoint_config = EndpointCoreConfigInput(
    served_entities=[served_entity],
)

try:
    # create_and_wait blocks until the endpoint reaches READY state
    print(f"Deploying endpoint '{endpoint_name}' — this may take a few minutes...")
    endpoint = w.serving_endpoints.create_and_wait(
        name=endpoint_name,
        config=endpoint_config,
    )
    print(f"\n\u2705 Serving endpoint deployed: {endpoint_name}")
    print(f"   State  : {endpoint.state.ready}")
    print(f"   URL    : {w.config.host}/serving-endpoints/{endpoint_name}/invocations")
except Exception as e:
    err_str = str(e)
    if "already exists" in err_str.lower():
        print(f"\u26a0\ufe0f  Endpoint '{endpoint_name}' already exists — updating to new model version...")
        endpoint = w.serving_endpoints.update_config_and_wait(
            name=endpoint_name,
            served_entities=[served_entity],
        )
        print(f"✅ Serving endpoint updated: {endpoint_name}")
        print(f"   State  : {endpoint.state.ready}")
    else:
        print(f"Endpoint error: {e}")
        raise

## Test Inference

Send a sample candidate payload to the live endpoint and verify the hire/no-hire prediction is returned correctly.

In [ ]:
# ---------------------------------------------------------------------------
# Test real-time inference via the serving endpoint
# Sample candidate: Sarah Chen (C001 — top-scoring candidate)
# All 8 features required; new candidates supply interview_score + culture_fit
# from the hiring interview when calling the agent.
# ---------------------------------------------------------------------------
import requests, time

host    = w.config.host.rstrip("/")
token   = w.config.token
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

sample_input = {
    "dataframe_records": [
        {
            "education_score":          85,
            "experience_score":         90,
            "leadership_score":         85,
            "certification_score":      95,
            "skills_match_score":       92,
            "industry_relevance_score": 95,
            "interview_score":          88,
            "culture_fit":              88,
        }
    ]
}

print(f"Querying endpoint '{endpoint_name}' with Sarah Chen's features...")
try:
    resp = requests.post(
        f"{host}/serving-endpoints/{endpoint_name}/invocations",
        headers=headers,
        json=sample_input,
        timeout=60,
    )
    resp.raise_for_status()
    predictions = resp.json().get("predictions", resp.json())
    pred_label  = predictions[0] if isinstance(predictions, list) else predictions
    result_text = "HIRE" if pred_label == 1 else "DO NOT HIRE"
    print(f"\n✅ Inference result for Sarah Chen")
    print(f"   Raw prediction : {pred_label}")
    print(f"   Recommendation : {result_text}")
    print(f"   Full response  : {resp.json()}")
except Exception as e:
    print(f"⚠️  Inference test failed: {e}")
    print("   The endpoint may still be warming up. Re-run this cell in 60 seconds.")

## Summary & Next Steps

### What was accomplished

| Step | Result |
|---|---|
| Data loaded | `candidates` Silver table — 10 training-eligible rows (C001–C010) |
| Features | 8 scoring dimensions including `culture_fit` |
| Models trained | LogisticRegression, RandomForestClassifier, GradientBoostingClassifier |
| Experiment tracked | MLflow — all runs visible in the Experiments UI |
| Best model selected | Highest AUC-ROC (see comparison table above) |
| UC registration | `bx4.hrd_2030.hr_predictive_hiring` @champion |
| Serving endpoint | `hr-predictive-hiring-endpoint` (scale-to-zero, Small) |
| Inference tested | Sarah Chen — C001 |

### How new candidates get scored
C011–C020 are missing `skills_match_score`, `interview_score`, and `culture_fit` (not yet interviewed).
The HireRight Agent accepts these values from the hiring manager at runtime and calls the endpoint for
a real-time hire/no-hire prediction alongside the probability score.

### Model Explainability Note
The RandomForest and GradientBoosting runs log `feature_importances.json` to their MLflow artifacts,
allowing stakeholders to understand which scoring dimensions drive the hiring prediction.
Typically **experience_score** and **leadership_score** carry the highest weight, consistent
with the Director of HR role requirements.

**Proceed to** `06_evaluate_register_agent.ipynb` to wire the predictive model, Genie Space, and vector search
into the **HireRight AI Agent** powered by `databricks-gpt-5-4`.